# Cuisine_rating.csv — Cleaning Log

Guided, non-destructive cleaning. The original `Cuisine_rating.csv` is only ever read. All fixes are applied to `Cuisine_rating_clean.csv`, one confirmed decision at a time. Each decision below lives in its own cell, in order, so any single decision can be undone by re-running the notebook up to (but not including) that cell.

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Original: read-only, ground truth for the inventory below.
df_original = pd.read_csv('Cuisine_rating.csv')
df_original.columns = [c.strip() for c in df_original.columns]

# Working copy: every fix from here on is applied to this dataframe, then saved to Cuisine_rating_clean.csv.
df = pd.read_csv('Cuisine_rating_clean.csv')
df.columns = [c.strip() for c in df.columns]

print('Original shape:', df_original.shape)
print('Working copy shape:', df.shape)

Original shape: (200, 15)
Working copy shape: (200, 15)


## 1. Shape & dtype overview

In [2]:
df_original.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   User ID         200 non-null    int64  
 1   Area code       200 non-null    int64  
 2   Location        200 non-null    str    
 3   Gender          200 non-null    str    
 4   YOB             200 non-null    int64  
 5   Marital Status  200 non-null    str    
 6   Activity        200 non-null    str    
 7   Budget          200 non-null    int64  
 8   Cuisines        200 non-null    str    
 9   Alcohol         200 non-null    str    
 10  Smoker          200 non-null    str    
 11  Food Rating     200 non-null    int64  
 12  Service Rating  200 non-null    int64  
 13  Overall Rating  200 non-null    float64
 14  Often A S       200 non-null    str    
dtypes: float64(1), int64(6), str(8)
memory usage: 23.6 KB


## 2. Duplicates inventory

In [3]:
print('Exact duplicate rows:', df_original.duplicated().sum())
print('Duplicate User IDs:', df_original['User ID'].duplicated().sum())

# Near-duplicate: identical on every column except the ID
id_col = 'User ID'
subset_cols = [c for c in df_original.columns if c != id_col]
near_dupes = df_original[df_original.duplicated(subset=subset_cols, keep=False)].sort_values(subset_cols)
print(f'\nNear-duplicate rows (identical on all columns except {id_col}): {len(near_dupes)}')
near_dupes

Exact duplicate rows: 0
Duplicate User IDs: 0

Near-duplicate rows (identical on all columns except User ID): 2


,User ID,Area code,Location,Gender,YOB,Marital Status,Activity,Budget,Cuisines,Alcohol,Smoker,Food Rating,Service Rating,Overall Rating,Often A S
28,29,154,"Market City, NY",Female,1977,Married,Student,4,French,Socially,Never,2,1,1.5,No
128,129,154,"Market City, NY",Female,1977,Married,Student,4,French,Socially,Never,2,1,1.5,No


No exact duplicate rows and no duplicate `User ID`s. But **User ID 29 and User ID 129 are identical on every other column** (same Area code, Location, demographics, Cuisines, and both ratings) — a strong near-duplicate. Queued for Step 3.

## 3. Missing values inventory

In [4]:
missing = pd.DataFrame({
    'missing_count': df_original.isna().sum(),
    'missing_pct': (df_original.isna().mean() * 100).round(2)
}).sort_values('missing_count', ascending=False)
missing

,missing_count,missing_pct
User ID,0,0.0
Area code,0,0.0
Location,0,0.0
Gender,0,0.0
YOB,0,0.0
Marital Status,0,0.0
Activity,0,0.0
Budget,0,0.0
Cuisines,0,0.0
Alcohol,0,0.0


No missing values in any column (0% across the board). All columns are under the 20% threshold, so none are queued for a Step 4 decision — nothing to fix here.

## 4. Outlier / impossible-value scan

In [5]:
numeric_cols = ['Area code', 'YOB', 'Budget', 'Food Rating', 'Service Rating', 'Overall Rating']
for col in numeric_cols:
    s = df_original[col]
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((s < lo) | (s > hi)).sum()
    print(f'{col}: IQR bounds=({lo:.1f}, {hi:.1f}) -> {n_out} statistical outliers')

print()
print('Domain-impossible checks:')
print('  YOB outside [1920, 2026]:', ((df_original["YOB"] < 1920) | (df_original["YOB"] > 2026)).sum())
print('  Budget outside [1, 5]:', ((df_original["Budget"] < 1) | (df_original["Budget"] > 5)).sum())
print('  Food Rating outside [1, 5]:', ((df_original["Food Rating"] < 1) | (df_original["Food Rating"] > 5)).sum())
print('  Service Rating outside [1, 5]:', ((df_original["Service Rating"] < 1) | (df_original["Service Rating"] > 5)).sum())
print('  Overall Rating outside [1, 5]:', ((df_original["Overall Rating"] < 1) | (df_original["Overall Rating"] > 5)).sum())
print('  Area code outside [100, 999]:', ((df_original["Area code"] < 100) | (df_original["Area code"] > 999)).sum())

Area code: IQR bounds=(70.5, 210.5) -> 0 statistical outliers
YOB: IQR bounds=(1927.5, 2043.5) -> 0 statistical outliers
Budget: IQR bounds=(0.0, 8.0) -> 0 statistical outliers
Food Rating: IQR bounds=(-2.5, 9.5) -> 0 statistical outliers
Service Rating: IQR bounds=(-2.5, 9.5) -> 0 statistical outliers
Overall Rating: IQR bounds=(0.2, 6.2) -> 0 statistical outliers

Domain-impossible checks:
  YOB outside [1920, 2026]: 0
  Budget outside [1, 5]: 0
  Food Rating outside [1, 5]: 0
  Service Rating outside [1, 5]: 0
  Overall Rating outside [1, 5]: 0
  Area code outside [100, 999]: 0


No statistical outliers (all numeric columns fall within 1.5×IQR of their quartiles) and no domain-impossible values (all ratings in range, YOB sane, Area code well-formed). Nothing queued for Step 5 — this pass found no issues to decide on.

## 5. Category consistency scan

In [6]:
import difflib

def norm(v):
    return ' '.join(v.strip().lower().replace(',', ', ').split())

for col in df_original.select_dtypes(include='object').columns:
    vals = df_original[col].unique().tolist()
    normed = {v: norm(v) for v in vals}
    seen = set()
    clusters_found = []
    for v1 in vals:
        if v1 in seen:
            continue
        cluster = [v1]
        for v2 in vals:
            if v2 == v1 or v2 in seen:
                continue
            ratio = difflib.SequenceMatcher(None, normed[v1], normed[v2]).ratio()
            if ratio > 0.85:
                cluster.append(v2)
        if len(cluster) > 1:
            for c in cluster:
                seen.add(c)
            clusters_found.append(cluster)
    if clusters_found:
        print(f'--- {col} ---')
        for cluster in clusters_found:
            counts = {c: (df_original[col] == c).sum() for c in cluster}
            print(f'  {counts}')

--- Location ---
  {'Upper East Side,NY': np.int64(30), 'Upper West Side,NY': np.int64(18)}
  {'Central Park,NY': np.int64(24), 'Central Park,ny': np.int64(8)}
  {'Market City, NY': np.int64(20), 'Market City, MY': np.int64(2)}


/var/folders/sz/3_np9r_91db6_dhc9819tkwh0000gn/T/ipykernel_81726/1089480691.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_original.select_dtypes(include='object').columns:


`Location` surfaces three findings:

1. **`Central Park,NY` (24) vs `Central Park,ny` (8)** — casing difference, same real location. Real cluster, queued for Step 6.
2. **`Market City, NY` (20) vs `Market City, MY` (2)** — one-letter difference, looks like a typo (NY → MY) rather than a real second city. Real cluster, queued for Step 6.
3. **`Upper East Side,NY` (30) vs `Upper West Side,NY` (18)** — the similarity checker flagged this pair, but East Side and West Side are genuinely different neighborhoods. **False positive — dismissed, not queued.**

Separately (not caught by the similarity check, since there's no competing spelling to cluster against): `China Town, NY` and `Cedar Hill, NY` have a space after the comma, while most other `Location` values don't (`St. George,NY`, `Upper East Side,NY`, etc.). This doesn't merge any duplicate categories, but it's an inconsistent format worth asking about — queued as a small Step 6 item.

## 6. Dtype audit

In [7]:
df_original.dtypes

User ID             int64
Area code           int64
Location              str
Gender                str
YOB                 int64
Marital Status        str
Activity              str
Budget              int64
Cuisines              str
Alcohol               str
Smoker                str
Food Rating         int64
Service Rating      int64
Overall Rating    float64
Often A S             str
dtype: object

| Column | Current dtype | Inferred "should be" | Reasoning |
|---|---|---|---|
| Area code | int64 | category / string | It's a location code, not a measurable quantity — summing or averaging it is meaningless, and it's 1:1 with `Location`. |
| Gender, Marital Status, Activity, Cuisines, Alcohol, Smoker, Often A S, Location | object (str) | category | Low cardinality (2–10 distinct values each), repeated many times — `category` is the semantically correct pandas dtype and is more memory-efficient. |
| User ID | int64 | int64 (no change) | Sequential unique identifier; arithmetic on it isn't meaningful either, but it's already effectively used only for lookups/joins, and converting a 1..200 unique key to `category` buys nothing. Left as-is. |
| YOB | int64 | int64 (no change) | Only a year is given (no month/day), so a full `datetime` dtype would imply false precision. Plain int is the honest representation. |
| Budget, Food Rating, Service Rating | int64 | int64 (no change) | Ordinal 1–5 scales; kept numeric since they're used in arithmetic (means, correlations) throughout the analysis. |
| Overall Rating | float64 | float64 (no change) | Correct already — derived as a .5-step average of Food and Service Rating. |

Two real decisions queued for Step 7: `Area code` → category/string, and the batch of low-cardinality text columns → `category`.

# Decisions Log

Everything below is applied to the **working copy** (`df` / `Cuisine_rating_clean.csv`) only, one confirmed decision at a time.

### Decision 1 — near-duplicate rows (User ID 29 / 129)

**User choice: keep both, treat as a false positive / genuine coincidence.** No change applied.

In [8]:
# Decision 1: keep both User ID 29 and 129 — user confirmed this is a false positive,
# not a true duplicate. No rows dropped, df unchanged.
print('Rows kept:', df.shape[0])

Rows kept: 200


### Decision 2 — category consistency: `Location` "Central Park" casing

**User choice: standardize on `Central Park,NY`** (majority spelling, 24 of 32 rows).

In [9]:
# Decision 2: standardize 'Central Park,ny' -> 'Central Park,NY'
n_before = (df['Location'] == 'Central Park,ny').sum()
df['Location'] = df['Location'].replace({'Central Park,ny': 'Central Park,NY'})
df.to_csv('Cuisine_rating_clean.csv', index=False)
print(f'{n_before} rows updated')
df['Location'].value_counts()

8 rows updated


Location
St. George,NY         46
Central Park,NY       32
Upper East Side,NY    30
Riverdale,NY          28
China Town, NY        22
Market City, NY       20
Upper West Side,NY    18
Market City, MY        2
Cedar Hill, NY         2
Name: count, dtype: int64

### Decision 3 — category consistency: `Location` "Market City" typo

**User choice: standardize on `Market City, NY`** (majority spelling; "MY" treated as a typo, not a real second city).

In [10]:
# Decision 3: standardize 'Market City, MY' -> 'Market City, NY'
n_before = (df['Location'] == 'Market City, MY').sum()
df['Location'] = df['Location'].replace({'Market City, MY': 'Market City, NY'})
df.to_csv('Cuisine_rating_clean.csv', index=False)
print(f'{n_before} rows updated')
df['Location'].value_counts()

2 rows updated


Location
St. George,NY         46
Central Park,NY       32
Upper East Side,NY    30
Riverdale,NY          28
China Town, NY        22
Market City, NY       22
Upper West Side,NY    18
Cedar Hill, NY         2
Name: count, dtype: int64

### Decision 4 — category consistency: `Location` comma-spacing format

**User choice: standardize all values to no space after the comma** (`City,NY`), matching the majority pattern.

In [11]:
# Decision 4: normalize comma-spacing in Location to 'City,ST' (no space)
import re
n_before = df['Location'].str.contains(', ').sum()
df['Location'] = df['Location'].apply(lambda v: re.sub(r',\s+', ',', v))
df.to_csv('Cuisine_rating_clean.csv', index=False)
print(f'{n_before} rows updated')
df['Location'].value_counts()

46 rows updated


Location
St. George,NY         46
Central Park,NY       32
Upper East Side,NY    30
Riverdale,NY          28
China Town,NY         22
Market City,NY        22
Upper West Side,NY    18
Cedar Hill,NY          2
Name: count, dtype: int64

### Decision 5 — dtype: `Area code`

**User choice: convert to `category`.** It's a location code, not a quantity to do arithmetic on, and maps 1:1 with `Location`.

Note: CSV has no native dtype metadata, so this conversion round-trips as a plain int/string once reloaded from `Cuisine_rating_clean.csv` — it's captured here as the correct dtype to apply when loading the CSV for analysis (e.g. `pd.read_csv(..., dtype={'Area code': 'category'})`).

In [12]:
# Decision 5: convert Area code to category dtype (in-memory) and re-save
df['Area code'] = df['Area code'].astype('category')
print(df['Area code'].dtype, '-', df['Area code'].cat.categories.size, 'categories')
df.to_csv('Cuisine_rating_clean.csv', index=False)

category - 65 categories


### Decision 6 — dtype: low-cardinality text columns → `category`

**User choice: convert all 8 to `category`** — `Gender`, `Marital Status`, `Activity`, `Cuisines`, `Alcohol`, `Smoker`, `Often A S`, `Location`.

Same CSV caveat as Decision 5: these round-trip as plain strings once reloaded from the CSV; apply `.astype('category')` (or a `dtype=` map on `read_csv`) when loading for analysis.

In [13]:
# Decision 6: convert the low-cardinality text columns to category dtype (in-memory) and re-save
category_cols = ['Gender', 'Marital Status', 'Activity', 'Cuisines', 'Alcohol', 'Smoker', 'Often A S', 'Location']
for c in category_cols:
    df[c] = df[c].astype('category')
df.to_csv('Cuisine_rating_clean.csv', index=False)
df.dtypes

User ID              int64
Area code         category
Location          category
Gender            category
YOB                  int64
Marital Status    category
Activity          category
Budget               int64
Cuisines          category
Alcohol           category
Smoker            category
Food Rating          int64
Service Rating       int64
Overall Rating     float64
Often A S         category
dtype: object

## Wrap-up

In [14]:
decision_log = pd.DataFrame([
    {'column/rows': 'User ID 29, 129', 'issue type': 'near-duplicate rows', 'decision': 'kept both', 'rationale': 'user judged it a genuine coincidence / false positive, not a true duplicate'},
    {'column/rows': 'Location', 'issue type': 'category consistency (casing)', 'decision': "'Central Park,ny' -> 'Central Park,NY'", 'rationale': 'majority spelling (24 of 32 rows)'},
    {'column/rows': 'Location', 'issue type': 'category consistency (typo)', 'decision': "'Market City, MY' -> 'Market City, NY'", 'rationale': 'one-letter typo, majority spelling (20 of 22 rows)'},
    {'column/rows': 'Location', 'issue type': 'category consistency (formatting)', 'decision': "removed space after comma, e.g. 'China Town, NY' -> 'China Town,NY'", 'rationale': 'matched majority no-space format across the column'},
    {'column/rows': 'Area code', 'issue type': 'dtype', 'decision': 'int64 -> category', 'rationale': "it's a location code, not a quantity; 1:1 with Location"},
    {'column/rows': 'Gender, Marital Status, Activity, Cuisines, Alcohol, Smoker, Often A S, Location', 'issue type': 'dtype', 'decision': 'object -> category', 'rationale': 'low-cardinality repeated text, semantically categorical'},
])
decision_log

,column/rows,issue type,decision,rationale
0,"User ID 29, 129",near-duplicate rows,kept both,user judged it a genuine coincidence / false p...
1,Location,category consistency (casing),"'Central Park,ny' -> 'Central Park,NY'",majority spelling (24 of 32 rows)
2,Location,category consistency (typo),"'Market City, MY' -> 'Market City, NY'","one-letter typo, majority spelling (20 of 22 r..."
3,Location,category consistency (formatting),"removed space after comma, e.g. 'China Town, N...",matched majority no-space format across the co...
4,Area code,dtype,int64 -> category,"it's a location code, not a quantity; 1:1 with..."
5,"Gender, Marital Status, Activity, Cuisines, Al...",dtype,object -> category,"low-cardinality repeated text, semantically ca..."


Each decision lives in its own numbered cell above, in order, so any single decision can be reverted by re-running the notebook up to (but not including) that cell and continuing from there — no need to restart the whole cleaning process to undo one choice.

**`Cuisine_rating.csv` (the original) was never modified.** All fixes were applied to `Cuisine_rating_clean.csv`, which now has: the `Location` casing/typo/formatting fixes applied, and `Area code` plus the 8 low-cardinality text columns as `category` dtype (note: reload with `.astype('category')` since CSV doesn't persist dtype — see Decisions 5 & 6). Row count is unchanged at 200 (the near-duplicate pair was kept).